[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/word_embeddings_glove.ipynb)

# Pretrained word embeddings: GloVe

[GloVe](https://nlp.stanford.edu/projects/glove/) (Global Vectors) embeddings were learned from the co-occurrences of words in a corpus of 6 billion words (Wikipedia and news). Unlike the tiny corpus of the [word_embeddings_cbow](word_embeddings_cbow.ipynb) notebook, they capture rich relations between words, which we can explore with similarities and analogies.

## Set the dependencies and download

We start by importing the dependencies and downloading the _GloVe_ embeddings (a ZIP file of ~800 MB, with embeddings of 50, 100, 200 and 300 dimensions). We only extract the 50 dimensions file.

In [ ]:
import os
import urllib.request
import zipfile

# Directory for GloVe
glove_dir = "glove"
zip_path = os.path.join(glove_dir, "glove.6B.zip")

# File we need (the ZIP also has the 100, 200 and 300 dimensions files)
glove_name = "glove.6B.50d.txt"
glove_file = os.path.join(glove_dir, glove_name)

# 1. Create directory if missing
os.makedirs(glove_dir, exist_ok=True)

if os.path.exists(glove_file):
    print("GloVe text file already extracted — skipping download.")
else:
    # 2. Download the ZIP file, if missing
    if not os.path.exists(zip_path):
        print("GloVe ZIP not found. Downloading…")
        url = "https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip"
        urllib.request.urlretrieve(url, zip_path)
        print("Download complete.")

    # 3. Extract only the file we need
    print("Extracting", glove_name, "…")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extract(glove_name, glove_dir)
    print("Extraction complete.")

## Load and use the embeddings

Load the 50 dimension vector embedding: each line of the file has a word followed by the 50 values of its vector (400000 words).

In [ ]:
import numpy as np

def load_glove(path):
    word2vec = {}
    with open(path, "r", encoding="utf8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            word = parts[0]
            vector = np.array(parts[1:], dtype=np.float32)
            word2vec[word] = vector
    return word2vec

glove50 = load_glove(glove_file)
print("Loaded vocabulary size:", len(glove50))

## Build the matrix and index

The vectors are normalized (length 1), so that the dot product between two vectors is their cosine similarity.

In [ ]:
import torch

# Build a matrix and index mapping
vocab = list(glove50.keys())
word_to_idx = {w: i for i, w in enumerate(vocab)}

embedding_matrix = torch.from_numpy(np.stack([glove50[w] for w in vocab]))

embedding_matrix = embedding_matrix / embedding_matrix.norm(dim=1, keepdim=True)
print(embedding_matrix.shape)

## Some auxiliary functions

To find the most similar words (with the highest cosine similarity)

In [ ]:
def get_vector(word):
    if word not in word_to_idx:
        raise ValueError(f"'{word}' not in vocabulary")
    return embedding_matrix[word_to_idx[word]]

def most_similar(word, top_k=10):
    v = get_vector(word)
    sims = torch.mv(embedding_matrix, v)
    best = torch.topk(sims, top_k + 1).indices.tolist()
    best = [i for i in best if vocab[i] != word][:top_k]
    print(f"\nWords most similar to '{word}':")
    for i in best:
        print(f"{vocab[i]:>10s}   (cosine={sims[i]:.3f})")

In [ ]:
most_similar("king")
most_similar("paris")
most_similar("apple")

## Find analogies

$b - a + c$: for example, king - man + woman should be close to queen.

In [ ]:
def analogy(a, b, c, top_k=5):
    for w in [a, b, c]:
        if w not in word_to_idx:
            print(f"'{w}' not in vocab")
            return
    va = get_vector(a)
    vb = get_vector(b)
    vc = get_vector(c)

    query = vb - va + vc
    # Normalize
    query = query / query.norm()

    sims = torch.mv(embedding_matrix, query)
    best = torch.topk(sims, top_k + 3).indices.tolist()
    exclude = {word_to_idx[w] for w in [a, b, c]}
    best = [i for i in best if i not in exclude][:top_k]

    print(f"\nAnalogy: {a} → {b} :: {c} → ?")
    for i in best:
        print(f"{vocab[i]:>10s}   (cosine={sims[i]:.3f})")

In [ ]:
analogy("man", "king", "woman")
analogy("paris", "france", "rome")
analogy("big", "bigger", "small")
analogy("walk", "walked", "swim")

## Visualize the embeddings

Project the 50-dimensional vectors of some words to 2 dimensions with PCA. Note how the groups (royalty, places, fruits) are separated. Compare with the plot of the CBOW notebook, trained with a tiny corpus.

In [ ]:
import matplotlib.pyplot as plt


def plot_embeddings(vectors, words, title):
    """Projects the vectors to 2D with PCA and plots them with the words as labels."""
    centered = vectors - vectors.mean(dim=0)
    # the first 2 principal components (directions of largest variance)
    _, _, V = torch.linalg.svd(centered, full_matrices=False)
    points = centered @ V[:2].T

    plt.figure(figsize=(8, 8))
    plt.scatter(points[:, 0], points[:, 1])
    for (x, y), word in zip(points.tolist(), words):
        plt.annotate(word, (x, y), xytext=(3, 3), textcoords="offset points")
    plt.title(title)
    plt.show()


words = ["king", "queen", "man", "woman", "prince", "princess",
         "france", "paris", "italy", "rome", "portugal", "lisbon", "germany", "berlin",
         "apple", "orange", "banana", "grape"]
plot_embeddings(torch.stack([get_vector(w) for w in words]), words, "GloVe embeddings (PCA)")